# Welcome to the 2026 AIMS PINN Workshop — Julia example

This short notebook shows you around: where your files live, how to use the shared course materials, how Julia environments are set up here, and where to save data.


## Your folders

- **`~`** (your home, e.g. `/home/efs/<you>`) — your **persistent** workspace on EFS. **Save your work here.** It is the *same* on the CPU and GPU hubs, backed up daily, with a **20 GB** quota.
- **`~/course-materials`** — a **read-only** copy of the course GitHub repo ([open-AIMS/Julia_PINN_training_2026](https://github.com/open-AIMS/Julia_PINN_training_2026)), refreshed automatically. To change a file, **copy it into your home first**.
- **`~/my_work`** — (created below) a good place to keep outputs.
- Don't rely on `/tmp` — it is wiped. Don't try to write into `~/course-materials` — it is read-only.


## 0. Where am I?


In [ ]:
println("Home directory : ", homedir())
println("Working dir    : ", pwd())
println("Julia version  : ", VERSION)
println("Depot path     : ", DEPOT_PATH)  # writable: your ~/.julia (Pkg.add + caches land here), then the shared read-only course depot
println("Load path      : ", Base.load_path())  # your personal env stacked on top of the shared course env

## 1. The shared course materials

Everything from the course GitHub repo is available read-only at `~/course-materials`.


In [ ]:
materials = joinpath(homedir(), "course-materials")
@assert isdir(materials) "course-materials not found — it appears on login"
println("Course materials live in: ", materials, "\n")
for name in first(sort(readdir(materials)), 20)
    println("  ", name)
end

## 2. Your environment, and using the course packages

Your session uses **two stacked environments**: a **shared, read-only course environment** (Lux, NeuralPDE, ModelingToolkit, Plots, …, preinstalled) with your **personal environment on top**. 
So you can just `using` the course packages — no install needed. 
Here we use a couple of standard-library packages so the demo runs even before the course stack is finalised:


In [ ]:
using LinearAlgebra, Statistics, Random
Random.seed!(42)
A = rand(3, 3)
println("A =")
show(stdout, "text/plain", A); println()
println("\nEigenvalues : ", eigvals(A))
println("Column means: ", vec(mean(A, dims=1)))

## 3. Saving your data

Save anything you want to keep in your **home** — for example the **`~/my_work`** folder (the one the file browser opens into), as the cell below does. Your home is on EFS — persistent + backed up. Mind the 20 GB quota.


In [ ]:
outdir  = joinpath(homedir(), "my_work"); mkpath(outdir)
outfile = joinpath(outdir, "hello_julia.txt")
open(outfile, "w") do io
    println(io, "Hello from Julia on the AIMS PINN hub!")
end
println("Saved to : ", outfile)
println("Contents : ", read(outfile, String))

## 4. GPU check (GPU hub)

The **GPU hub** gives you an NVIDIA GPU — the login page and the JupyterLab top bar show *which one* (it can vary day to day, but your code never changes). Julia talks to it through **`CUDA.jl`**, preinstalled in the course environment.

The cell below runs a small GPU computation when a GPU is present, and explains itself on the CPU hub.

In [ ]:
# GPU check — runs a real test on the GPU hub; explains gracefully on the CPU hub.
using CUDA
if CUDA.functional()
    println("✅ GPU is active: ", CUDA.name(CUDA.device()))
    x = CUDA.rand(2000, 2000)
    y = x * x                          # matrix multiply on the GPU
    CUDA.@sync y
    println("   GPU matmul OK — sum(x*x) = ", sum(y))
    CUDA.@sync x * x                   # warm-up
    tg = CUDA.@elapsed CUDA.@sync x * x
    xc = Array(x); tc = @elapsed xc * xc
    println("   GPU ≈ ", round(tg*1e3, digits=1), " ms   vs   CPU ≈ ", round(tc*1e3, digits=1), " ms")
else
    println("ℹ️  No active GPU here — you're on the CPU hub (or the GPU isn't up).")
    println("   Run this on the GPU hub (gpu.aims.accumulationpoint.com, 8am–8pm AEST) to use the GPU.")
end

## 5. PINN smoke test — push the machine 🚀

A tiny **physics-informed neural network (PINN)** that solves the 1-D Poisson equation

$$-u''(x) = \sin(\pi x), \qquad u(0)=u(1)=0,$$

whose exact solution is $u(x) = \sin(\pi x)/\pi^2$. This exercises the **full SciML stack** (NeuralPDE, ModelingToolkit, Optimization, Lux) end-to-end — a good "is everything installed, and is it fast?" check.

The first run takes **about a minute** — roughly ~20 s to load the SciML packages plus ~50 s to compile the first solve. **Re-running the cell is instant.** (Plotting is instant thanks to the course **system image**; the SciML compile is one-time per session, so don't worry if it sits for a bit.)

In [ ]:
using NeuralPDE, ModelingToolkit, Optimization, OptimizationOptimJL, Lux, Random, DomainSets
using Plots

@parameters x
@variables u(..)
Dxx = Differential(x)^2
eq  = Dxx(u(x)) ~ -sinpi(x)                 # -u'' = sin(πx)
bcs = [u(0) ~ 0.0, u(1) ~ 0.0]
dom = [x ∈ DomainSets.Interval(0.0, 1.0)]

chain = Lux.Chain(Lux.Dense(1 => 16, tanh), Lux.Dense(16 => 1))
disc  = PhysicsInformedNN(chain, GridTraining(0.01))
@named pde = PDESystem(eq, bcs, dom, [x], [u(x)])
prob = discretize(pde, disc)

println("training PINN (first run compiles ~1 min; re-runs are instant) ...")
t = @elapsed res = Optimization.solve(prob, OptimizationOptimJL.BFGS(); maxiters=400)
println("trained in ", round(t, digits=1), " s   |   final loss = ", res.objective)

phi    = disc.phi
xs     = 0:0.01:1
upred  = [first(phi([xi], res.u)) for xi in xs]
uexact = sinpi.(xs) ./ pi^2
println("max abs error vs exact = ", round(maximum(abs.(upred .- uexact)), digits=5))

plot(xs, uexact, label="exact", lw=3)
plot!(xs, upred, label="PINN", ls=:dash, lw=2,
      title="PINN smoke test:  -u'' = sin(pi x)", xlabel="x", ylabel="u(x)")

## 6. Installing your own package

Anything **you** add goes into **your personal environment** in your home (`~/.julia`) — it **persists** across sessions and across both hubs, and it does **not** touch the shared course base. 
Add a registered package **by name** (resolved from the General registry; the first run precompiles). *(Do this last — adding a package re-resolves your personal environment, which can make the next `using NeuralPDE` recompile once.)* Here we add [`DistributionsFactories.jl`](https://github.com/Distribution-Matching/DistributionsFactories.jl), which builds probability distributions that match moments (mean, variance, …) or quantiles you specify:


In [ ]:
import Pkg
Pkg.add("DistributionsFactories")

In [ ]:
using DistributionsFactories
using Distributions

# DistributionsFactories builds a distribution that matches the moments you ask for.
# Example: take a Normal, truncate it to the interval [-1, 4], and let the package
# solve the underlying Normal so the *truncated* distribution has mean 1.0 and std 0.8.
d = make_dist(truncated(Normal(); lower=-1, upper=4), mean=1.0, std=0.8)

println("distribution      : ", d)
println("requested moments : mean = 1.0, std = 0.8")
println("achieved moments  : mean = ", round(mean(d), digits=4),
        ", std = ", round(std(d), digits=4))
println("parent Normal solved to : ", d.untruncated)

In [ ]:
import Pkg; Pkg.status()   # your personal environment: the packages you've added on top of the course base

---
Happy modelling! Duplicate this notebook (right-click → *Duplicate*), or copy files out of `~/course-materials`, to start experimenting.
